# COMSOC — construcción del dataset

Corre el pipeline completo: 15 Excel → `data/processed/comsoc_polizas.parquet`.
Tarda ~1 minuto. Hazlo una vez y trabaja sobre el parquet.

**Kernel: `pnt_analysis`** (environment de conda, local). Selecciónalo arriba a la
derecha en VS Code. Si no aparece, reinicia la ventana: el `ipykernel` ya está instalado.

In [ ]:
import comsoc
from comsoc.config import RAW_DIR, ROOT

print('comsoc', comsoc.__version__)
print('proyecto:', ROOT)
print('excel crudos:', len(list(RAW_DIR.glob('*.xlsx'))), '(deben ser 15)')

In [ ]:
from comsoc import build

df = build.construir(base=2020)
df.head()

## Pruebas de aceptación

Si alguna falla, **no sigas al análisis**: la cifra está mal y todo lo que
construyas encima hereda el error. Ver `PLAN_MIGRACION.md` §1.6.

In [ ]:
from comsoc import validate

validate.reporte(df)

## Identificadores

| Columna | Qué identifica |
|---|---|
| `poliza_id` | la **póliza**: hash de año + grupo de partida + entidad + número de póliza |
| `renglon_id` | el **renglón**. Único en todo el dataset |
| `n_renglones` | cuántos renglones tiene la póliza de esta fila |
| `ocurrencia` | desempata las filas idénticas (0, 1, 2…) |

Son hash del contenido, no contadores: **estables entre corridas y entre máquinas**.

`poliza_id` **no** incluye `vintage` a propósito: la misma póliza en la edición
preliminar y en la definitiva de 2023 recibe el mismo id, y eso es lo que permite
compararlas.

In [ ]:
from comsoc import ids

print(ids.verificar_ids(df))   # colisiones debe ser 0

# Las pólizas más grandes de la serie
(df.groupby(['poliza_id', 'anio_fuente', 'institucion'], as_index=False)
   .agg(renglones=('renglon_id', 'size'), monto=('monto_total', 'sum'))
   .nlargest(10, 'monto'))

## Sesiones posteriores

Ya no repitas la ingesta: carga el parquet.

In [ ]:
import pandas as pd
from comsoc.config import POLIZAS_PARQUET

df = pd.read_parquet(POLIZAS_PARQUET)

# Serie principal: ediciones definitivas, sin intercambios en especie
serie = df[(df['vintage'] == 'definitiva') & (~df['es_intercambio'])]

(serie.groupby(['anio_fuente', 'partida_grupo'])['monto_total'].sum() / 1e6).round(1)

## Exportar

El parquet ya está en `data/processed/`. Para compartir con quien no use Python:

In [ ]:
from comsoc import export

export.exportar(df, formatos=('csv',))